# Model-based Reinforcement Learning

<img src="https://raw.githubusercontent.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students/refs/heads/main/assets/logo.jpg?raw=true" style="float: left; width: 15%" />

[CSC_53439_EP-2026](https://moodle.ip-paris.fr/course/view.php?id=19695) Lab session #3

2019-2026 Jérémie Decock

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeremiedecock/polytechnique-csc-53439-ep-2026-students/blob/main/lab3_mcts.ipynb)

[![My Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jeremiedecock/polytechnique-csc-53439-ep-2026-students/main?filepath=lab3_mcts.ipynb)

[![NbViewer](https://raw.githubusercontent.com/jupyter/design/master/logos/Badges/nbviewer_badge.svg)](https://nbviewer.jupyter.org/github/jeremiedecock/polytechnique-csc-53439-ep-2026-students/blob/main/lab3_mcts.ipynb)

[![Local](https://img.shields.io/badge/Local-Save%20As...-blue)](https://github.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students/raw/main/lab3_mcts.ipynb)

## Introduction

In our preceding labs, we've delved into model-free approaches, encompassing both value-based and policy-based methods.

Model-free reinforcement learning methods learn directly from episodes of experience. They are called "model-free" because they do not need to know or learn the model of the environment (i.e., the reward and transition probability functions). Examples include Q-learning and policy gradients.

On the other hand, model-based reinforcement learning methods use a model of the environment (i.e., the reward and transition probability functions) to make decisions: this model is either given (e.g., the rules of a board game) or learned from experience. They can plan ahead by simulating future states with the model, which can lead to more efficient learning than model-free methods.

The aim of this lab is to provide an in-depth exploration of one of the most famous model-based reinforcement learning methods: *Monte Carlo Tree Search (MCTS)*.
We will focus on the planning aspect of model-based reinforcement learning (using MCTS), so we provide the model of the environment (*NoughtsAndCrossesState* and *NoughtsAndCrossesAction*), but in many practical cases, the model is not available and must be learned from experience.

In this Python notebook, you will implement and assess this algorithm on the Noughts and Crosses board game, one step of MCTS at a time:
- Exercise 1: the rollout policy and the "simulation" step;
- Exercise 2: the "expansion" step;
- Exercise 3: the "backpropagation" step;
- Exercise 4: the tree policy (UCB) and the "selection" step;
- Exercise 5: the complete MCTS search, used to play a full game.

You can either:
- open, edit and execute the notebook in *Google Colab* following this link: https://colab.research.google.com/github/jeremiedecock/polytechnique-csc-53439-ep-2026-students/blob/main/lab3_mcts.ipynb ; this is the **recommended** choice as you have nothing to install on your computer
- open, edit and execute the notebook in *MyBinder* (if for any reason the Google Colab solution doesn't work): https://mybinder.org/v2/gh/jeremiedecock/polytechnique-csc-53439-ep-2026-students/main?filepath=lab3_mcts.ipynb
- download, edit and execute the notebook on your computer if Python3 and JupyterLab are already installed: https://github.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students/raw/main/lab3_mcts.ipynb

If you work with Google Colab or MyBinder, **remember to save or download your work regularly or you may lose it!**

## Lab Submission

Please submit your completed notebook in [Moodle: "Lab 3 - Submission"](https://moodle.ip-paris.fr/mod/assign/view.php?id=432460).

### Submission Guidelines

1. **File Naming:** Rename your notebook as follows: **`firstname_lastname-03.ipynb`** where `firstname` and `lastname` match your email address. *Example: `jesse_read-03.ipynb`*
2. **Clear Output Cells:** To reduce file size (**must be under 500 KB**), clear all output cells before submitting. This includes rendered images, videos, plots, and dataframes...
   - **JupyterLab:**
     - Click **"Kernel" → "Restart Kernel and Clear Outputs of All Cells..."**
     - Then go to **"File" → "Save Notebook As..."**
   - **Google Colab:**
     - Click **"Edit" → "Clear all outputs"**
     - Then go to **"File" → "Download" → "Download .ipynb"**
   - **VSCode:**
     - Click **"Clear All Outputs"**
     - Then **save your file**
3. **Upload Your File:** Only **`.ipynb`** files are accepted.

**Note:** Bonus parts (if any) are optional, as their name suggests.

## Setup the Python environment

This notebook only relies on the Python standard library and the [Graphviz](https://graphviz.org/) `dot` command line tool to draw the MCTS trees.
A complete list of Python dependencies can be found in the following [requirements_lab3.txt](https://raw.githubusercontent.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students/main/requirements_lab3.txt) file.

### If you use Google Colab

If you use Google Colab, there is nothing to install: Graphviz and all the required Python libraries are already available.

### If you have downloaded the notebook on your computer and execute it in your own Python environment

To set up the necessary dependencies, run the following commands to establish a [Python virtual environment (venv)](https://docs.python.org/3/library/venv.html) that includes all the essential libraries for this lab.

#### On POSIX systems (Linux, macOS, WSL, ...)

```bash
python3 -m venv env-lab3
source env-lab3/bin/activate
python3 -m pip install --upgrade pip
python3 -m pip install -r https://raw.githubusercontent.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students/main/requirements_lab3.txt
```

#### On Windows

```bat
python -m venv env-lab3
env-lab3\Scripts\activate.bat
python -m pip install --upgrade pip
python -m pip install -r https://raw.githubusercontent.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students/main/requirements_lab3.txt
```

These commands are for the Command Prompt (`cmd.exe`). In PowerShell, activate the virtual environment with `env-lab3\Scripts\Activate.ps1` instead.

#### Install Graphviz

Graphviz is not a Python library: it has to be installed separately (see https://graphviz.org/download/).
For instance:
- on Debian GNU/Linux, Ubuntu, ...: `sudo apt install graphviz`
- on macOS (with [Homebrew](https://brew.sh/)): `brew install graphviz`
- on Windows: download and run the installer available at https://graphviz.org/download/ (and check that the `bin` directory of Graphviz is added to your `PATH`)

### Run CSC-53439-EP notebooks locally in a VSCode Dev Container

If you use [Visual Studio Code](https://code.visualstudio.com/) and have Docker (or Podman) installed on your computer, the course repository includes a ready-to-use [Dev Container](https://code.visualstudio.com/docs/devcontainers/containers) configuration (in the `.devcontainer/` directory) that provides Python 3.12, Graphviz and all the libraries required for this lab.

1. Install the [Dev Containers](https://marketplace.visualstudio.com/items?itemName=ms-vscode-remote.remote-containers) extension in VSCode.
   If you use Podman instead of Docker, set the `dev.containers.dockerPath` setting to `podman` in VSCode.
2. Clone the repository and open it in VSCode:
   ```bash
   git clone https://github.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students.git
   code polytechnique-csc-53439-ep-2026-students
   ```
3. When VSCode asks, click **"Reopen in Container"** (or open the Command Palette with `Ctrl+Shift+P` / `Cmd+Shift+P` and run **"Dev Containers: Reopen in Container"**).
   The first start may take several minutes, as the image is built and the Python dependencies listed in `requirements.txt` are installed automatically.
4. Open this notebook and select the container's Python interpreter as the kernel: **"Select Kernel" → "Python Environments..." → "Python 3.12"**.

The repository folder on your computer is shared with the container, so your work is kept when the container is stopped or rebuilt.

### Check that Graphviz is installed

The next cell should print the version of Graphviz installed on your system.

In [ ]:
!dot -V

### Import required packages

In [ ]:
from collections import deque
from copy import deepcopy
import math
from pathlib import Path
import random
import subprocess
from typing import Any, Callable, Dict, List, Optional

from IPython.display import Image, display   # To display Graphviz images in the notebook

In [ ]:
FIGS_DIR = Path("figs/") / "lab3"       # Where to save figures (.dot and .png files)

In [ ]:
FIGS_DIR.mkdir(parents=True, exist_ok=True)

## Monte Carlo Tree Search for Noughts and Crosses (Tic-Tac-Toe)

### The MCTS algorithm

Monte Carlo Tree Search (MCTS) is a model-based reinforcement learning algorithm that uses Monte Carlo simulation to evaluate actions in a decision-making problem. It's particularly effective in complex domains such as games, where it's not feasible to compute an exact solution.

Every time a decision is required, the MCTS algorithm performs the following steps for a specified number of simulations:
1. **Selection**: Start from the root node (the current state) and select successive child nodes according to a *tree policy*, until reaching a node that is not fully expanded (i.e., that has untried actions) or a terminal node.
2. **Expansion**: If this node is not a terminal node (i.e., it does not end the game), then create one (or more) child node(s) for untried action(s) and select one of them.
3. **Simulation**: From the new node, run a simulated play of the game according to a *rollout policy*, either to the end of the game or to a predetermined depth limit.
4. **Backpropagation**: Use the result of the game simulation to update the statistics of the nodes on the path from the new node back to the root.

Once the computational budget is exhausted, the action leading to the "best" child of the root is chosen (e.g., the child with the highest average reward, or the most visited child).

<img src="https://github.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students/raw/refs/heads/main/assets/mcts.png" />

(source: Richard S. Sutton and Andrew G. Barto: "Reinforcement Learning: An Introduction")

Here is a pseudo-code version of the algorithm (the names of the functions are those of the Python functions you will implement in this lab):

```
function mcts_search(root):
    while within computational budget:
        node = select_node(root)
        reward = simulate(node)
        backpropagate(node, reward)
    return the action leading to get_best_child(root, exploration_ratio=0)

function select_node(node):
    while node is not terminal:
        if node is not fully expanded:
            return expand(node)
        else:
            node = get_best_child(node, exploration_ratio)
    return node

function expand(node):
    choose an untried action of node
    create the corresponding new child node
    return the new child node

function simulate(node):
    play a game from the state of node using the rollout policy (e.g. random moves)
    return the final reward

function backpropagate(node, reward):
    while node is not null:
        update the statistics of node with reward
        node = parent of node

function get_best_child(node, exploration_ratio):
    return the child of node with the highest UCB value (computed with exploration_ratio)
```

Here's a brief explanation of each part:

- **MCTS search**: It runs the main loop of the algorithm. Within the computational budget (like a time limit or a maximum number of iterations), it selects nodes, simulates games from those nodes, and then backpropagates the results.

- **Select node**: This function descends the tree from the root, choosing among the children of fully expanded nodes with the UCB formula (this is the UCT algorithm, for *Upper Confidence bounds applied to Trees*), until it reaches a node that has untried actions (which is then expanded) or a terminal node.

- **Expand**: If the node has any untried actions, it creates a new child node for one of these actions.

- **Simulate**: From the new node, it simulates a random game (or a game with some heuristic-based decisions), often until a terminal state of the game is reached.

- **Backpropagate**: Once the simulation is complete, the result (the final reward) is propagated back up the tree, updating the statistics (number of visits and total reward) of each node along the path from the new node to the root.

- **Best child**: When the computational budget is exhausted, the action leading to the child of the root with the highest average reward for the player to move (i.e. the highest UCB value without exploration) is chosen. Other criteria are possible, like choosing the most visited child of the root.

For further reading, see the bibliography at the end of this notebook (in particular the survey of Browne et al. (2012) and section 8.11 of Sutton and Barto's book).

### Exercise 1: The rollout policy and the "simulation" step

Let's begin by implementing the rollout policy (also known as the playout policy or the default policy), which is used during the simulation phase (the third phase of the MCTS algorithm).

The next two cells define the Action and State classes of the [Noughts and Crosses environment](https://en.wikipedia.org/wiki/Tic-tac-toe) (also known as Tic-tac-toe). For this lab, we'll forgo the use of the Gymnasium framework in favor of a custom, tailored environment.

Note that in this environment:
- rewards are always given from the point of view of player X: the reward of a terminal state is +1 if X wins, -1 if O wins and 0 for a draw;
- an action is identified by the (row, column) indices of the cell where the player puts its mark, from (0, 0) (top left cell) to (2, 2) (bottom right cell): for instance, the action `(0, 2)` puts a mark in the top right cell.

In [ ]:
class NoughtsAndCrossesAction():
    """
    Represents an action in the game of Noughts and Crosses (Tic-Tac-Toe).

    Attributes
    ----------
    player : int
        The player making the action. 1 for 'X', -1 for 'O'.
    x : int
        The row index of the action on the game board (0 for the top row).
    y : int
        The column index of the action on the game board (0 for the left column).
    """

    def __init__(self, player: int, x: int, y: int):
        self.player = player
        self.x = x
        self.y = y

    def __str__(self) -> str:
        return str((self.x, self.y))

    def __repr__(self) -> str:
        return str(self)

    def __eq__(self, other: Any) -> bool:
        return self.__class__ == other.__class__ and self.x == other.x and self.y == other.y and self.player == other.player

    def __hash__(self) -> int:
        return hash((self.x, self.y, self.player))

In [ ]:
class NoughtsAndCrossesState():
    """
    Represents the state of the game of Noughts and Crosses (Tic-Tac-Toe).

    Attributes
    ----------
    PLAYER_SYMBOLS : dict of int to str
        A dictionary mapping the player numerical values used internally to their symbols ('X' or 'O').
    board : list of list of int
        The game board represented as a 2D list (1 for 'X', -1 for 'O', 0 for an empty cell).
    current_player : int
        The current player making a move. 1 for 'X', -1 for 'O'.
    """

    PLAYER_SYMBOLS = {
        1: "X",
        -1: "O"
    }

    def __init__(self, board: Optional[List[List[int]]] = None, current_player: int = 1):
        self.board: List[List[int]] = board if board is not None else [[0, 0, 0], [0, 0, 0], [0, 0, 0]]
        self.current_player: int = current_player

    def getCurrentPlayer(self) -> int:
        """
        Get the current player making a move.

        Returns
        -------
        int
            The current player. 1 for 'X', -1 for 'O'.
        """
        return self.current_player

    def getPossibleActions(self) -> List[NoughtsAndCrossesAction]:
        """
        Get a list of possible actions that the current player can take.

        Returns
        -------
        list of NoughtsAndCrossesAction
            A list of possible actions that the current player can take.
        """
        possible_actions = []
        for i in range(len(self.board)):
            for j in range(len(self.board[i])):
                if self.board[i][j] == 0:
                    possible_actions.append(NoughtsAndCrossesAction(player=self.current_player, x=i, y=j))
        return possible_actions

    def takeAction(self, action: NoughtsAndCrossesAction) -> "NoughtsAndCrossesState":
        """
        Take an action and return the resulting state.

        Parameters
        ----------
        action : NoughtsAndCrossesAction
            The action to take.

        Returns
        -------
        NoughtsAndCrossesState
            The resulting state after taking the action.
        """
        new_state = deepcopy(self)
        new_state.board[action.x][action.y] = action.player
        new_state.current_player = self.current_player * -1
        return new_state

    def getWinner(self) -> int:
        """
        Get the winner of the game.

        Returns
        -------
        int
            1 if 'X' has won, -1 if 'O' has won, 0 otherwise (draw or game not over).
        """
        size = len(self.board)
        lines = [list(row) for row in self.board]                           # Rows
        lines += [list(column) for column in zip(*self.board)]              # Columns
        lines.append([self.board[i][i] for i in range(size)])               # Diagonal
        lines.append([self.board[i][size - i - 1] for i in range(size)])    # Anti-diagonal
        for line in lines:
            if abs(sum(line)) == 3:
                return sum(line) // 3
        return 0

    def isTerminal(self) -> bool:
        """
        Check if the current state is a terminal state (game over).

        Returns
        -------
        bool
            True if the state is terminal (one player has won or the board is full), False otherwise.
        """
        return self.getWinner() != 0 or all(cell != 0 for row in self.board for cell in row)

    def getReward(self) -> float:
        """
        Get the reward for the current state, from the point of view of player 'X'.

        Returns
        -------
        float
            1.0 if 'X' has won, -1.0 if 'O' has won, 0.0 otherwise (draw or non-terminal state).
        """
        return float(self.getWinner())

    def graphviz_str(self) -> str:
        """
        Get a compact representation of the board, used to draw MCTS trees with Graphviz.

        Returns
        -------
        str
            The board, one row per line ('x' and 'o' for the two players, '_' for empty cells).
        """
        output_str = ""
        for row in self.board:
            output_str += ''.join([NoughtsAndCrossesState.PLAYER_SYMBOLS[x].lower() if x in (-1, 1) else '_' for x in row]) + '\n'
        return output_str

    def __str__(self) -> str:
        output_str = f"Current player: {NoughtsAndCrossesState.PLAYER_SYMBOLS[self.current_player]}\n"
        for row in self.board:
            output_str += ' '.join([NoughtsAndCrossesState.PLAYER_SYMBOLS[x] if x in (-1, 1) else '_' for x in row]) + '\n'
        return output_str

In the context of Monte Carlo Tree Search (MCTS), the `random_policy` is used during the simulation phase of the algorithm. It's a type of rollout policy where actions are selected randomly from the set of available actions in each state.

The purpose of this rollout policy is to provide an estimate of the value of a state by simulating the outcome of the game from that state to the end, under the assumption that both players play randomly.
This estimate is then used to update the value of the state in the MCTS tree. It guides the selection of actions in future simulations by the *tree policy* during the selection phase (step 1) of MCTS.

**Task 1.1**: Implement the `random_policy` function.

In [ ]:
def random_policy(state: NoughtsAndCrossesState) -> NoughtsAndCrossesAction:
    """
    Select a random action from the possible actions in the current state.

    Parameters
    ----------
    state : NoughtsAndCrossesState
        The current state of the game.

    Returns
    -------
    NoughtsAndCrossesAction
        A randomly selected action from the possible actions in the current state.
    """
    # action = TODO... # Select an action uniformly at random among the possible actions of the current state (see `state.getPossibleActions()` and `random.choice`)

    return action

Now let's use this rollout policy in the simulate function.

The `simulate` function in Monte Carlo Tree Search (MCTS) is used to perform the simulation phase of the algorithm.

In this phase, the algorithm plays out a complete game or a certain number of steps from the current state, according to a specified policy (in this case, `rollout_policy`). The policy is typically a simple, fast-to-compute policy, such as choosing actions uniformly at random.

The purpose of the simulation is to obtain an estimate of the value of the current state. This estimate is then used to update the values of the states and actions in the MCTS tree, which guides the selection of actions in future simulations.

The `simulate` function has to continue to select actions and transition to new states until it reaches a terminal state (a state that ends the game). It then has to return the reward of the terminal state (+1 if X wins, -1 if O wins, 0 for a draw), which serves as the estimate of the value of the initial state.

**Task 1.2**: Implement the `simulate` function.

In [ ]:
def simulate(
    state: NoughtsAndCrossesState,
    rollout_policy: Callable[[NoughtsAndCrossesState], NoughtsAndCrossesAction],
    verbose: bool = False
) -> float:
    """
    Simulate a game from the current state, using the rollout policy, and return the reward value.

    This function performs the simulation phase of the Monte Carlo Tree Search (MCTS) algorithm.

    Parameters
    ----------
    state : NoughtsAndCrossesState
        The current state of the game.
    rollout_policy : Callable[[NoughtsAndCrossesState], NoughtsAndCrossesAction]
        The policy used to select actions during the simulated game.
    verbose : bool
        Print all traversed states and taken actions.

    Returns
    -------
    float
        The reward value at the end of the simulated game (from the point of view of player 'X').
    """
    if verbose:
        print(f"Initial state:\n{state}")

    while not state.isTerminal():

        # action = TODO...# Select an action with the rollout policy
        # state = TODO... # Take this action to get the new state (see `state.takeAction`)

        if verbose:
            print(f"Action: {action}")
            print(f"New state:\n{state}")

    return state.getReward()

Now execute the following cell to check if your implementation works as intended.

**Task 1.3**: Test the `random_policy` and the `simulate` functions.

In [ ]:
state = NoughtsAndCrossesState()

reward = simulate(state, random_policy, verbose=True)

print(f"Reward: {reward}")

### Exercise 2: The "expansion" step

We will implement the `expand` function, but first, we define the `TreeNode` class. The `TreeNode` class is a data structure that represents a node in the MCTS tree.
In this deterministic environment, each node represents a state in the problem's state space, and each edge represents an action that transitions between states.

Here are the attributes of our `TreeNode` class:

- `state`: The state in the state space that this node represents.
- `parent`: The parent node of this node in the MCTS tree. The root node has a `parent` of `None`.
- `children`: A dictionary mapping actions to child nodes. If an action leads from this node's state to another state, then `children[action]` is the `TreeNode` for that state.
- `is_terminal`: A boolean flag indicating whether this node's state is a terminal state (i.e., it ends the game).
- `is_fully_expanded`: A boolean flag indicating whether all possible actions from this node's state have been added to the `children` dictionary (it is always `True` for terminal nodes).
- `num_visits`: The number of times this node has been visited during the MCTS algorithm.
- `total_reward`: The total reward received during simulations that passed through this node (from the point of view of player X). This is used to compute the average reward of this node, which is `total_reward / num_visits`.

In [ ]:
class TreeNode():
    """
    Represents a node in the Monte Carlo Tree Search (MCTS) algorithm.

    Attributes
    ----------
    state : NoughtsAndCrossesState
        The state of the game at this node.
    is_terminal : bool
        True if the state is a terminal state, False otherwise.
    is_fully_expanded : bool
        True if all possible actions have been expanded (always True for a terminal node), False otherwise.
    parent : TreeNode
        The parent node of this node (None for the root node).
    num_visits : int
        The number of times this node has been visited.
    total_reward : float
        The total reward accumulated at this node (from the point of view of player 'X').
    children : dict of NoughtsAndCrossesAction to TreeNode
        A dictionary mapping actions to child nodes.
    """

    def __init__(
        self,
        state: NoughtsAndCrossesState,
        parent: Optional["TreeNode"]
    ):
        self.state = state
        self.is_terminal = state.isTerminal()
        self.is_fully_expanded = self.is_terminal
        self.parent = parent
        self.num_visits = 0
        self.total_reward = 0.0
        self.children: Dict[NoughtsAndCrossesAction, "TreeNode"] = {}

    def __str__(self) -> str:
        s = []
        s.append(f"total_reward: {self.total_reward}")
        s.append(f"num_visits: {self.num_visits}")
        s.append(f"is_terminal: {self.is_terminal}")
        s.append(f"expanded_actions: {list(self.children.keys())}")
        return f"{self.__class__.__name__}: " + ', '.join(s)

    def to_graphviz(
        self,
        save_file_path: Path,
        exploration_ratio: Optional[float] = None,
        max_depth: Optional[int] = None
    ) -> str:
        """
        Write the tree rooted at the current node in a Graphviz (DOT) file.

        Each node of the tree is drawn as a box containing `total_reward/num_visits`
        followed by the board of its state.

        Parameters
        ----------
        save_file_path : Path
            The path of the Graphviz file to write.
        exploration_ratio : float, optional
            If not None, also display the UCB value of each node (computed with this exploration ratio, using the `ucb` function of exercise 4).
        max_depth : int, optional
            If not None, only draw the nodes up to this depth (the current node has depth 0).

        Returns
        -------
        str
            The content of the Graphviz file.
        """

        dot_str = "digraph G {\n"
        dot_str += '    node [shape=box, fontname="Courier"];\n'

        # Traverse the tree breadth-first from the current node, and add each node and the edge from its parent to the graph.
        # Each node gets a unique Graphviz identifier, so that different nodes with the same label are not merged.
        node_ids: Dict[TreeNode, int] = {}
        queue = deque([(self, 0)])     # (node, depth) pairs

        while len(queue) > 0:
            # Retrieve the first element of the queue
            node, depth = queue.popleft()
            node_ids[node] = len(node_ids)

            label = f"{node.total_reward:g}/{node.num_visits}"
            if exploration_ratio is not None and node.parent is not None and node.num_visits > 0:
                label += f"\\nUCB: {ucb(node, exploration_ratio):.3f}"
            label += "\\n" + node.state.graphviz_str().replace("\n", "\\n")
            dot_str += f'    {node_ids[node]} [label="{label}"];\n'

            if node is not self:
                # Add the edge from the parent node to the current node
                dot_str += f"    {node_ids[node.parent]} -> {node_ids[node]};\n"

            # Add child nodes into the queue (unless the maximum depth is reached)
            if max_depth is None or depth < max_depth:
                queue.extend((child, depth + 1) for child in node.children.values())

        dot_str += "}\n"

        with open(save_file_path, "w") as f:
            f.write(dot_str)

        return dot_str

The `display_tree` function, defined in the next cell, will be used in the following exercises to draw an MCTS tree with Graphviz and to display it in the notebook.

Each box of the drawn tree is a node: it shows the `total_reward/num_visits` of the node followed by the board of its state (`x` and `o` for the two players, `_` for empty cells).
Two optional parameters can be used:
- `exploration_ratio`: if it is set, the UCB value of each node is also displayed (this requires the `ucb` function of exercise 4);
- `max_depth`: if it is set, only the nodes up to this depth are drawn (this is useful for large trees).

In [ ]:
def display_tree(node: TreeNode, file_name: str, exploration_ratio: Optional[float] = None, max_depth: Optional[int] = None) -> Image:
    """
    Draw the tree rooted at `node` with Graphviz and display it in the notebook.

    The tree is saved in `FIGS_DIR` as a Graphviz file (`file_name.dot`) and as a PNG image (`file_name.png`).

    Parameters
    ----------
    node : TreeNode
        The root of the tree to draw.
    file_name : str
        The name of the saved files (without extension).
    exploration_ratio : float, optional
        If not None, also display the UCB value of each node (computed with this exploration ratio).
    max_depth : int, optional
        If not None, only draw the nodes up to this depth (`node` has depth 0).

    Returns
    -------
    Image
        The PNG image of the tree (displayed in the notebook when it is returned by the last line of a cell).
    """
    dot_file_path = FIGS_DIR / f"{file_name}.dot"
    png_file_path = FIGS_DIR / f"{file_name}.png"

    node.to_graphviz(save_file_path=dot_file_path, exploration_ratio=exploration_ratio, max_depth=max_depth)
    subprocess.run(["dot", "-Tpng", str(dot_file_path), "-o", str(png_file_path)], check=True)   # Requires Graphviz

    return Image(filename=png_file_path)

The `expand` function in Monte Carlo Tree Search (MCTS) is used to perform the expansion phase of the algorithm.

In this phase, the algorithm chooses a node of the current MCTS tree that is not fully expanded yet, and creates a child node corresponding to one of the actions that have not been tried yet from the state represented by this node. This new child node is then used for the simulation phase.

The purpose of the expansion phase is to grow the MCTS tree by exploring new states of the game. This allows the algorithm to gradually build up a more and more accurate representation of the value of each state and action, as more simulations are performed.

Each call to the `expand` function adds **one** child node to the given node:
- if the node is terminal or already fully expanded (i.e., `node.is_fully_expanded` is `True`), raise a `ValueError` (this should never happen in MCTS);
- choose the first action returned by `node.state.getPossibleActions()` that has not been tried yet (i.e., that is not yet a key of the `node.children` dictionary); in practice, an untried action is often chosen at random, but this deterministic choice makes the tests below reproducible;
- create the corresponding child node (its state is the state obtained by taking this action, and its parent is `node`) and add it to `node.children`;
- if all the possible actions of the current state have now been tried, set `node.is_fully_expanded` to `True`;
- return the new child node.

**Task 2.1**: Implement the `expand` function.

In [ ]:
def expand(node: TreeNode) -> TreeNode:
    """
    Expand the given node by adding a child node for an action that has not been tried yet.

    Parameters
    ----------
    node : TreeNode
        The node to expand.

    Returns
    -------
    TreeNode
        The newly added child node.
    """
    # TODO...

Now execute the following cells to check if your implementation works as intended.

You should obtain a root node with 3 children (X plays in (0, 0), (0, 1) or (0, 2)): the second child is a terminal state (X wins) and the first child has 2 children.

**Task 2.2**: Test the `expand` function (the tree is drawn with the `display_tree` function defined above).

In [ ]:
root_state = NoughtsAndCrossesState(board=[[0, 0, 0], [-1, 1, 1], [-1, 1, -1]], current_player=1)
root_node = TreeNode(root_state, None)

new_node1 = expand(root_node)
new_node2 = expand(root_node)
new_node3 = expand(root_node)

new_node11 = expand(new_node1)
new_node12 = expand(new_node1)

In [ ]:
display_tree(root_node, "lab3_mcts_test")

In [ ]:
print(f"Number of children of the root node: {len(root_node.children)} (expected: 3)")
print(f"Number of children of new_node1: {len(new_node1.children)} (expected: 2)")

assert len(root_node.children) == 3 and root_node.is_fully_expanded, "The root node should have 3 children and be fully expanded"
assert all(child.parent is root_node for child in root_node.children.values()), "The parent of the children of the root node should be the root node"
assert new_node2.is_terminal, "new_node2 should be a terminal node (X wins)"
assert len(new_node1.children) == 2 and new_node1.is_fully_expanded, "new_node1 should have 2 children and be fully expanded"
assert len(new_node3.children) == 0 and not new_node3.is_fully_expanded, "new_node3 should have no child and should not be fully expanded"

print("expand passed the tests")

### Exercise 3: The "backpropagation" step

The `backpropagate` function in Monte Carlo Tree Search is used to perform the backpropagation phase of the algorithm.

In this phase, after a simulation has been performed and a reward has been obtained, the algorithm updates the information in the nodes on the path from the root to the leaf node that was expanded. This typically involves incrementing the `num_visits` attribute of each node on the path, and adding the reward to the `total_reward` attribute of each node.

The purpose of the backpropagation phase is to propagate the information obtained from the simulation back up the tree, so that it can guide the selection of actions in future simulations.

In your implementation, the `backpropagate` function takes as input the node from which the simulation has been performed and the reward obtained from the simulation. It has to update the `num_visits` and `total_reward` attributes of this node and of all its ancestors up to the root node (use the `parent` attribute to go up the tree).

Since rewards are always given from the point of view of player X, the same reward is added to all the nodes of the path, whatever the player who plays in these nodes (there is no need to change its sign): the point of view of each player will be taken into account in the UCB formula (exercise 4).

**Task 3.1**: Implement the `backpropagate` function.

In [ ]:
def backpropagate(node: TreeNode, reward: float) -> None:
    """
    Backpropagate the reward value from a node up to the root node.

    Parameters
    ----------
    node : TreeNode
        The node from which the simulation has been performed (i.e., the node returned by `select_node`).
    reward : float
        The reward value obtained from the simulation.
    """
    # TODO... # For each node on the path from `node` up to the root node (use the `parent` attribute, which is `None` for the root node), update statistics

Now execute the following cells to check if your implementation works as intended.

If you execute the cell that calls `backpropagate` several times, the statistics of the nodes are updated several times: in this case, execute again the cell that builds the tree first.

**Task 3.2**: Test the `backpropagate` function.

In [ ]:
root_state = NoughtsAndCrossesState(board=[[0, 0, 0], [-1, 1, 1], [0, 0, -1]], current_player=1)
root_node = TreeNode(root_state, None)

new_node1 = expand(root_node)
new_node2 = expand(root_node)
new_node3 = expand(root_node)
new_node4 = expand(root_node)
new_node5 = expand(root_node)

new_node11 = expand(new_node1)
new_node21 = expand(new_node2)
new_node22 = expand(new_node2)
new_node31 = expand(new_node3)

In [ ]:
display_tree(root_node, "lab3_mcts_test")

In [ ]:
backpropagate(new_node1, 0.)
backpropagate(new_node2, 1.)
backpropagate(new_node3, 1.)
backpropagate(new_node4, 0.)
backpropagate(new_node5, 0.)

backpropagate(new_node11, 0.)
backpropagate(new_node21, 1.)
backpropagate(new_node22, 0.)
backpropagate(new_node31, 0.)

In [ ]:
display_tree(root_node, "lab3_mcts_test")

In [ ]:
print(f"root_node: {root_node.total_reward:g}/{root_node.num_visits} (expected: 3/9)")
print(f"new_node2: {new_node2.total_reward:g}/{new_node2.num_visits} (expected: 2/3)")
print(f"new_node21: {new_node21.total_reward:g}/{new_node21.num_visits} (expected: 1/1)")

assert (root_node.total_reward, root_node.num_visits) == (3., 9), "The statistics of the root node should be 3/9"
assert (new_node2.total_reward, new_node2.num_visits) == (2., 3), "The statistics of new_node2 should be 2/3"
assert (new_node21.total_reward, new_node21.num_visits) == (1., 1), "The statistics of new_node21 should be 1/1"

print("backpropagate passed the tests")

### Exercise 4: The tree policy (UCB) and the "selection" step

The Upper Confidence Bound (UCB) formula is used during the selection phase (the first step) of Monte Carlo Tree Search.
In this phase, starting from the root node, the algorithm selects the best child node according to a certain criterion, and continues this process until it reaches a node that is not fully expanded (or a terminal node). The criterion for selecting the best child is typically the Upper Confidence Bound (UCB) formula, which balances the exploitation of child nodes with high average reward and the exploration of child nodes with few visits.

In this lab, the UCB value of a child node $a$ is defined as follows:

$$
\text{UCB}(a) = \frac{w_a}{n_a} + c \sqrt{\frac{2 \ln n}{n_a}}
$$

Where:

- $w_a$ is the total reward accumulated in child $a$, from the point of view of the player who chooses among the children, i.e. the player who plays in the parent node (the `total_reward` attribute of the child node is given from the point of view of player X: see the practical tips below).
- $n_a$ is the number of times child $a$ has been visited (the `num_visits` attribute of the child node).
- $n$ is the number of times the parent node has been visited (the `num_visits` attribute of the parent node).
- $c$ is the exploration parameter (the `exploration_ratio` parameter), which is a constant determining the balance between exploration and exploitation. With $c = 1$, this formula is the UCB1 formula of Auer et al. (2002). In this lab, we use $c = 1/\sqrt{2}$ by default: this is the value proposed by Kocsis and Szepesvári (2006) for rewards in $[0, 1]$. Note that our rewards are in $[-1, 1]$, a range twice as large: with the same value of $c$, exploration has relatively less weight.

This formula balances two aspects: the exploitation of known good moves (represented by $\frac{w_a}{n_a}$), and the exploration of less-tried moves (represented by $c \sqrt{\frac{2 \ln n}{n_a}}$). The constant $c$ adjusts how much the algorithm favors exploration over exploitation.

**Task 4.1**: Implement the `ucb` function.

#### Practical tips

- Rewards are always given from the point of view of player X (+1 if X wins, -1 if O wins, 0 for a draw): `node.total_reward / node.num_visits` is the average reward *for player X*.
  However, the child is chosen by the player who plays in the state of the parent node: X wants to maximize this average reward, while O wants to minimize it.
  To make each player maximize its own average reward, multiply the exploitation term $\frac{w_a}{n_a}$ by the current player of the parent node, i.e. `node.parent.state.getCurrentPlayer()` (1 for X and -1 for O).
- The `ucb` function is only called on nodes that have been visited at least once ($n_a \geq 1$).
- Use `math.log` (natural logarithm) and `math.sqrt`.

In [ ]:
def ucb(node: TreeNode, exploration_ratio: float) -> float:
    """
    Calculate the Upper Confidence Bound (UCB) for a node.

    Parameters
    ----------
    node : TreeNode
        The node for which the UCB is calculated (its parent is `node.parent`).
    exploration_ratio : float
        The exploration ratio for balancing exploration and exploitation.

    Returns
    -------
    float
        The calculated UCB value.
    """
    # player = TODO... # Get the player who chooses among the children of `node.parent` (1 for X, -1 for O)

    # exploitation = TODO...# Compute the exploitation term
    # exploration = TODO... # Compute the exploration term

    # node_value = TODO... # Compute the UCB value

    return node_value

Now execute the following cells to check if your implementation works as intended (the UCB value of each visited node is now displayed in the tree).

**Task 4.2**: Test the `ucb` function.

In [ ]:
root_state = NoughtsAndCrossesState(board=[[0, 0, 0], [-1, 1, 1], [0, 0, -1]], current_player=1)
root_node = TreeNode(root_state, None)

new_node1 = expand(root_node)
new_node2 = expand(root_node)
new_node3 = expand(root_node)
new_node4 = expand(root_node)
new_node5 = expand(root_node)

new_node11 = expand(new_node1)
new_node21 = expand(new_node2)
new_node22 = expand(new_node2)
new_node31 = expand(new_node3)

backpropagate(new_node1, 0.)
backpropagate(new_node2, 1.)
backpropagate(new_node3, 1.)
backpropagate(new_node4, 0.)
backpropagate(new_node5, 0.)

backpropagate(new_node11, 0.)
backpropagate(new_node21, 1.)
backpropagate(new_node22, 0.)
backpropagate(new_node31, 0.)

In [ ]:
display_tree(root_node, "lab3_mcts_test", exploration_ratio=1/math.sqrt(2))

In [ ]:
exploration_ratio = 1 / math.sqrt(2)

# Children of the root node: X plays in the root node, so the exploitation term is the average reward of X
print(f"ucb(new_node1) = {ucb(new_node1, exploration_ratio=exploration_ratio):.4f} (expected: 1.0481)")
print(f"ucb(new_node2) = {ucb(new_node2, exploration_ratio=exploration_ratio):.4f} (expected: 1.5225)")

# Children of new_node2: O plays in new_node2, so the exploitation term is the average reward of O (i.e. the opposite of the average reward of X)
print(f"ucb(new_node21) = {ucb(new_node21, exploration_ratio=exploration_ratio):.4f} (expected: 0.0481)")
print(f"ucb(new_node22) = {ucb(new_node22, exploration_ratio=exploration_ratio):.4f} (expected: 1.0481)")

assert math.isclose(ucb(new_node1, exploration_ratio=exploration_ratio), 1.0481, abs_tol=1e-4)
assert math.isclose(ucb(new_node2, exploration_ratio=exploration_ratio), 1.5225, abs_tol=1e-4)
assert math.isclose(ucb(new_node21, exploration_ratio=exploration_ratio), 0.0481, abs_tol=1e-4), "Check the sign of the exploitation term when O plays in the parent node"
assert math.isclose(ucb(new_node22, exploration_ratio=exploration_ratio), 1.0481, abs_tol=1e-4)

print("ucb passed the tests")

Like the UCB formula, the `get_best_child` function is used during the selection phase of the algorithm.
The `get_best_child` function computes the UCB value for each child of the input node, and returns the child with the highest UCB value.

If several children have the same (highest) UCB value, one of them is chosen uniformly at random: put all these children in the list `best_nodes`, which is used by the last line of the function (`return random.choice(best_nodes)`).

With `exploration_ratio=0`, `get_best_child` returns the child with the highest average reward for the player who plays in the input node: this will be used at the end of the search to choose the action to play.

**Task 4.3**: Implement the `get_best_child` function.

In [ ]:
def get_best_child(node: TreeNode, exploration_ratio: float) -> TreeNode:
    """
    Get the best child node based on the Upper Confidence Bound (UCB) formula.

    Parameters
    ----------
    node : TreeNode
        The parent node.
    exploration_ratio : float
        The exploration ratio for balancing exploration and exploitation.

    Returns
    -------
    TreeNode
        The best child node.
    """
    # TODO... # Compute the UCB value of each child of `node` (with the `ucb` function)
    # best_nodes = TODO... # Make the list of the children having the highest UCB value

    return random.choice(best_nodes)

Now execute the following cells to check if your implementation works as intended.

**Task 4.4**: Test the `get_best_child` function.

In [ ]:
root_state = NoughtsAndCrossesState(board=[[0, 0, 0], [-1, 1, 1], [0, 0, -1]], current_player=1)
root_node = TreeNode(root_state, None)

new_node1 = expand(root_node)
new_node2 = expand(root_node)
new_node3 = expand(root_node)
new_node4 = expand(root_node)
new_node5 = expand(root_node)

new_node11 = expand(new_node1)
new_node21 = expand(new_node2)
new_node22 = expand(new_node2)
new_node31 = expand(new_node3)

backpropagate(new_node1, 0.)
backpropagate(new_node2, 1.)
backpropagate(new_node3, 1.)
backpropagate(new_node4, 0.)
backpropagate(new_node5, 0.)

backpropagate(new_node11, 0.)
backpropagate(new_node21, 1.)
backpropagate(new_node22, 0.)
backpropagate(new_node31, 0.)

In [ ]:
display_tree(root_node, "lab3_mcts_test", exploration_ratio=1/math.sqrt(2))

In [ ]:
exploration_ratio = 1 / math.sqrt(2)

best_node = get_best_child(root_node, exploration_ratio=exploration_ratio)
print(best_node)
print(best_node.state)

# The UCB values of the children of the root node are 1.0481, 1.5225, 1.5481, 1.4823 and 1.4823
assert best_node is new_node3, "The best child of the root node should be new_node3 (UCB ≈ 1.5481)"

# O plays in new_node2: its best child is new_node22 (UCB ≈ 1.0481, versus 0.0481 for new_node21)
assert get_best_child(new_node2, exploration_ratio=exploration_ratio) is new_node22, "The best child of new_node2 should be new_node22: check the sign of the exploitation term when O plays"

# Without exploration, the best child is the child with the highest average reward for the player who plays (X): new_node2 (2/3)
assert get_best_child(root_node, exploration_ratio=0.) is new_node2, "Without exploration, the best child of the root node should be new_node2"

print("get_best_child passed the tests")

The `select_node` function in Monte Carlo Tree Search (MCTS) is the main function used to perform the selection phase of the algorithm (and it also triggers the expansion phase).

Starting from the input node (the root of the tree), the `select_node` function iteratively moves down the tree:
- if the current node is terminal, it returns this node;
- if the current node is fully expanded, it moves to its best child according to the UCB criterion (using the `get_best_child` function);
- otherwise, it expands the current node (using the `expand` function) and returns the new child node.

The returned node is the node from which the simulation phase will be performed.

**Task 4.5**: Implement the `select_node` function.

In [ ]:
def select_node(node: TreeNode, exploration_ratio: float) -> TreeNode:
    """
    Select the node from which the next simulation will be performed (selection and expansion phases of MCTS).

    Parameters
    ----------
    node : TreeNode
        The current node in the tree.
    exploration_ratio : float
        The exploration ratio for selecting actions in the tree traversal phase.

    Returns
    -------
    TreeNode
        The selected node (a newly expanded node or a terminal node).
    """
    # TODO...

    return node

Now execute the following cells to check if your implementation works as intended.

The root node is fully expanded: `select_node` should move to its best child `new_node3` (see the previous test), which is not fully expanded, and return a new child of `new_node3`.

**Task 4.6**: Test the `select_node` function.

In [ ]:
root_state = NoughtsAndCrossesState(board=[[0, 0, 0], [-1, 1, 1], [0, 0, -1]], current_player=1)
root_node = TreeNode(root_state, None)

new_node1 = expand(root_node)
new_node2 = expand(root_node)
new_node3 = expand(root_node)
new_node4 = expand(root_node)
new_node5 = expand(root_node)

new_node11 = expand(new_node1)
new_node21 = expand(new_node2)
new_node22 = expand(new_node2)
new_node31 = expand(new_node3)

backpropagate(new_node1, 0.)
backpropagate(new_node2, 1.)
backpropagate(new_node3, 1.)
backpropagate(new_node4, 0.)
backpropagate(new_node5, 0.)

backpropagate(new_node11, 0.)
backpropagate(new_node21, 1.)
backpropagate(new_node22, 0.)
backpropagate(new_node31, 0.)

In [ ]:
display_tree(root_node, "lab3_mcts_test", exploration_ratio=1/math.sqrt(2))

In [ ]:
selected_node = select_node(root_node, exploration_ratio=1/math.sqrt(2))
print(selected_node)
print(selected_node.state)

assert selected_node.parent is new_node3, "The selected node should be a new child of new_node3"
assert selected_node.num_visits == 0, "The selected node should be a newly expanded node"

print("select_node passed the tests")

### Exercise 5: Putting it all together in the search function

Now, let's integrate all the components within the `search` method of the `MCTS` class: for each of the `num_simulations` iterations, select a node (`select_node`), simulate a game from this node (`simulate`) and backpropagate the result (`backpropagate`).

Once all simulations are performed, `search` returns the action leading to the child of the root with the highest average reward for the current player (i.e. `get_best_child` without exploration). Another common choice, often more robust, is the most visited child of the root.

**Task 5.1**: Implement the `MCTS.search` method.

In [ ]:
class MCTS():
    """
    Monte Carlo Tree Search (MCTS) algorithm for the game of Noughts and Crosses (Tic-Tac-Toe).

    Attributes
    ----------
    exploration_ratio : float
        The exploration ratio for selecting actions in the tree traversal phase.
    rollout_policy : Callable[[NoughtsAndCrossesState], NoughtsAndCrossesAction]
        The policy for simulating random games from a state.
    root : TreeNode or None
        The root of the tree built by the last call to `search` (None before the first call).
    """

    def __init__(self,
                 exploration_ratio: float = 1/math.sqrt(2),
                 rollout_policy: Callable[[NoughtsAndCrossesState], NoughtsAndCrossesAction] = random_policy):
        self.exploration_ratio = exploration_ratio
        self.rollout_policy = rollout_policy
        self.root: Optional[TreeNode] = None

    def search(self, current_state: NoughtsAndCrossesState, num_simulations: int = 1000) -> NoughtsAndCrossesAction:
        """
        Perform the Monte Carlo Tree Search (MCTS) algorithm to select the best action to take for the given state.

        Parameters
        ----------
        current_state : NoughtsAndCrossesState
            The current state of the game.
        num_simulations : int
            The number of simulations to perform.

        Returns
        -------
        NoughtsAndCrossesAction
            The best action to take according to the MCTS algorithm.
        """
        self.root = TreeNode(current_state, None)

        # for TODO... # Repeat `num_simulations` times
        #     node = TODO... # Select the node from which the simulation will be performed (selection and expansion steps)
        #     reward = TODO... # Simulate a game from the state of this node with the rollout policy (simulation step)
        #     TODO... # Backpropagate the reward from this node up to the root node (backpropagation step)

        # Choose the best child of the root without exploration (i.e. the child with the highest average reward for the current player)
        best_child = get_best_child(self.root, exploration_ratio=0.)

        # Return the action leading to this child
        best_action = next(action for action, child in self.root.children.items() if child is best_child)

        return best_action

Now complete the following cell so that MCTS plays a full game against itself (it chooses the moves of both X and O): at each move, use `mcts.search` to choose the action to play, and apply it to the current state.

As Tic-Tac-Toe is a draw when both players play optimally, MCTS playing against itself should (almost always) end in a draw. If X or O wins most of the games, check your implementation (in particular the sign of the exploitation term in the `ucb` function).

**Task 5.2**: Complete the following cell and test the MCTS algorithm.

In [ ]:
state = NoughtsAndCrossesState()

mcts = MCTS()

mcts_root_list = []   # The root of the MCTS tree built at each move (to display these trees below)

print(f"Initial state:\n{state}")

while not state.isTerminal():
    # action = TODO... # Choose the action to play with `mcts.search` (with 1000 simulations)
    # state = TODO... # Take this action to get the new state

    print(f"Action: {action}")
    print(f"New state:\n{state}")

    mcts_root_list.append(mcts.root)

print(f"Final reward: {state.getReward()} (1 if X wins, -1 if O wins, 0 for a draw)")

#### Display the MCTS tree at some moves of the game

The `i`-th tree of `mcts_root_list` (starting from 0) is the tree built by MCTS to choose the `i+1`-th move of the game.

The trees built for the first moves are too large to be drawn (up to 1000 nodes): the next cell only draws the trees built to choose the 5th to the 8th moves (if the game lasts that long), and limits their depth to `MAX_DEPTH` (set `MAX_DEPTH = None` to draw the whole trees).

In [ ]:
MAX_DEPTH = 2    # Maximum depth of the drawn trees (None to draw the whole trees)

for move_index in range(4, min(8, len(mcts_root_list))):
    print(f"Tree built by MCTS to choose move #{move_index + 1}:")
    display(display_tree(mcts_root_list[move_index], f"lab3_mcts_{move_index}", max_depth=MAX_DEPTH))

**Task 5.3**: Change the `exploration_ratio` (e.g. `MCTS(exploration_ratio=0.1)` or `MCTS(exploration_ratio=2.0)`), play a new game and watch the effect of this change on the drawn trees (you can set `MAX_DEPTH = None` to draw the whole trees) and on the outcome of the games. What do you observe?

**Answer:**

TODO...

## Bibliography

- Kocsis, Levente, and Csaba Szepesvári. "Bandit based monte-carlo planning." In European conference on machine learning, pp. 282-293. Berlin, Heidelberg: Springer Berlin Heidelberg, 2006. https://citeseerx.ist.psu.edu/document?repid=rep1&type=pdf&doi=6661e57237e4e8739b7a4946c4d3d4875376c068

- Browne, Cameron B., Edward Powley, Daniel Whitehouse, Simon M. Lucas, Peter I. Cowling, Philipp Rohlfshagen, Stephen Tavener, Diego Perez, Spyridon Samothrakis, and Simon Colton. "A survey of monte carlo tree search methods." IEEE Transactions on Computational Intelligence and AI in games 4, no. 1 (2012): 1-43. https://repository.essex.ac.uk/4117/1/MCTS-Survey.pdf

- Multi-armed Bandits: Richard S. Sutton and Andrew G. Barto: "Reinforcement Learning: An Introduction", chap.2 (http://incompleteideas.net/book/RLbook2020.pdf).

- Introduction to MCTS: Richard S. Sutton and Andrew G. Barto: "Reinforcement Learning: An Introduction", sect. 8.11, p.185 (http://incompleteideas.net/book/RLbook2020.pdf).

- Kamil Czarnogórski, "Monte Carlo Tree Search – beginners guide", 2018 https://int8.io/monte-carlo-tree-search-beginners-guide/

- Auer, Peter, Nicolo Cesa-Bianchi, and Paul Fischer. "Finite-time analysis of the multiarmed bandit problem." Machine learning 47 (2002): 235-256. https://link.springer.com/content/pdf/10.1023/A:1013689704352.pdf